**LOADING** **DATA**

In [ ]:
!pip install kagglehub

In [ ]:
!pip install transformers torch

In [ ]:
import kagglehub
path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

print("Path to dataset files:", path)

In [ ]:
import kagglehub

paths = kagglehub.dataset_download("grouplens/movielens-20m-dataset")

print("Path to dataset files:", paths)

In [ ]:
import os
os.listdir(path)

In [ ]:
print(os.listdir(paths))

In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')
movies_tmdb = pd.read_csv(path + "/tmdb_5000_movies.csv")
credits = pd.read_csv(path + "/tmdb_5000_credits.csv")


ratings = pd.read_csv(os.path.join(paths, "rating.csv"))
movies_movielens = pd.read_csv(os.path.join(paths, "movie.csv"))
links   = pd.read_csv(os.path.join(paths, "link.csv"))

# **preprocessing**

In [ ]:
print(ratings.shape)
print(movies_movielens.shape)
print(links.shape)
print(movies_tmdb.shape)

In [ ]:
ratings.head()

In [ ]:
movies_movielens.head()

In [ ]:
links.head()

In [ ]:
movies_tmdb.head()

In [ ]:
movies_tmdb[['id','title','overview']].head()

In [ ]:
print(ratings.isnull().sum())

In [ ]:
print(ratings.duplicated().sum())

In [ ]:
print(movies_movielens.isnull().sum())

In [ ]:
movies_movielens.duplicated().sum()

In [ ]:
movies_movielens['genres'] = movies_movielens['genres'].str.split('|')

In [ ]:
movies_movielens.head()

In [ ]:
movies_tmdb.isnull().sum()

In [ ]:
tmdb_selected = movies_tmdb[['id', 'title', 'overview', 'keywords',
                       'vote_average', 'vote_count', 'popularity',
                       'release_date', 'runtime', 'original_language']]

In [ ]:
import ast

def extract_names(text):
    try:
        items = ast.literal_eval(text)
        return [item['name'] for item in items]
    except:
        return []

tmdb_selected['keywords'] = tmdb_selected['keywords'].apply(extract_names)
print(tmdb_selected['keywords'].head())

In [ ]:
tmdb_selected['runtime'] = tmdb_selected['runtime'].fillna(tmdb_selected['runtime'].median())
tmdb_selected['release_date'] = pd.to_datetime(tmdb_selected['release_date'])
tmdb_selected['release_year'] = tmdb_selected['release_date'].dt.year
tmdb_selected['release_month'] = tmdb_selected['release_date'].dt.month
tmdb_selected = tmdb_selected.dropna(subset=['release_date'])
tmdb_selected = tmdb_selected.drop(columns=['release_date'])

In [ ]:
movies_tmdb.duplicated().sum()

In [ ]:
tmdb_selected = tmdb_selected.dropna(subset=['overview'])
tmdb_selected['overview'] =tmdb_selected['overview'].str.strip()

In [ ]:
tmdb_selected.head()

In [ ]:
links['tmdbId'] = links['tmdbId'].dropna().astype(int)
tmdb_selected['id'] = tmdb_selected['id'].astype(int)

In [ ]:
merged = links.merge(tmdb_selected,
                     left_on='tmdbId',
                     right_on='id')

In [ ]:
merged = merged.merge(movies_movielens[['movieId','genres']], on='movieId')

In [ ]:
print(merged.shape)


In [ ]:
print(merged.head())

In [ ]:
merged.isnull().sum()

In [ ]:
merged = merged.drop_duplicates(subset='movieId')

In [ ]:
merged = merged.drop(columns=['id', 'imdbId'])


In [ ]:
merged.head()

In [ ]:

valid_movies = merged['movieId'].unique()
ratings_filtered = ratings[ratings['movieId'].isin(valid_movies)]

In [ ]:
ratings_filtered = ratings_filtered.drop(columns=['timestamp'])

In [ ]:
ratings_filtered.head()

In [ ]:
print(ratings_filtered.shape)

**user watch history**

In [ ]:
transactions = ratings_filtered.groupby('userId')['movieId'].apply(list).reset_index()
transactions.columns = ['userId', 'movies']

print(transactions.head())

In [ ]:

movie_dict = merged.set_index('movieId')['title'].to_dict()


transactions['movies'] = transactions['movies'].apply(
    lambda x: [movie_dict.get(m, str(m)) for m in x]
)

print(transactions.head())

# **df for visualization extra**

In [ ]:
df = movies_tmdb

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.drop([
    'homepage',
    'tagline',
    'original_title',
    'overview',
    'keywords',
    'genres'
], axis=1, inplace=True)

In [ ]:
df['production_companies'] = df['production_companies'].apply(extract_names)
df['production_countries'] = df['production_countries'].apply(extract_names)
df['spoken_languages'] = df['spoken_languages'].apply(extract_names)

In [ ]:
df['runtime'] = df['runtime'].fillna(df['runtime'].median())
df = df.dropna(subset=['release_date'])

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

In [ ]:
df['release_date'] = pd.to_datetime(df['release_date'])
df['year'] = df['release_date'].dt.year
df['month'] = df['release_date'].dt.month

In [ ]:
df = df[df['budget'] > 0]
df = df[df['revenue'] > 0]
df = df[df['runtime'] > 0]
df['profit'] = df['revenue'] - df['budget']
df['roi'] = df['revenue'] / df['budget']
df['roi'] = df['roi'].replace([float('inf'), -float('inf')], 0)

# **Association rule Mining**



In [ ]:

!pip install mlxtend --quiet

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:

positive_ratings = ratings_filtered[ratings_filtered['rating'] >= 4.0].copy()


positive_ratings = positive_ratings.merge(
    merged[['movieId', 'title']],
    on='movieId',
    how='left'
)


positive_ratings = positive_ratings.dropna(subset=['title'])


movie_transactions = (
    positive_ratings
    .groupby('userId')['title']
    .apply(lambda x: sorted(list(set(x))))
    .reset_index()
)

movie_transactions.columns = ['userId', 'movies']


movie_transactions = movie_transactions[movie_transactions['movies'].apply(len) >= 2]

print("Number of users / transactions:", movie_transactions.shape[0])
print("Example transaction:")
movie_transactions.head()

In [ ]:

movie_frequency = positive_ratings['title'].value_counts()


TOP_N_MOVIES = 300
top_movies = set(movie_frequency.head(TOP_N_MOVIES).index)


filtered_transactions = movie_transactions['movies'].apply(
    lambda movies: [movie for movie in movies if movie in top_movies]
)


filtered_transactions = filtered_transactions[filtered_transactions.apply(len) >= 2].tolist()

print("Number of filtered transactions:", len(filtered_transactions))
print("Number of unique movies used:", len(top_movies))
print("Sample filtered transaction:")
filtered_transactions[0][:10]

In [ ]:

te = TransactionEncoder()
encoded_array = te.fit(filtered_transactions).transform(filtered_transactions)

encoded_movies = pd.DataFrame(encoded_array, columns=te.columns_)

print("Encoded data shape:", encoded_movies.shape)
encoded_movies.head()

In [ ]:

!pip install mlxtend --quiet

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:

positive_ratings = ratings_filtered[ratings_filtered['rating'] >= 4.0].copy()


positive_ratings = positive_ratings.merge(
    merged[['movieId', 'title']],
    on='movieId',
    how='left'
)


positive_ratings = positive_ratings.dropna(subset=['title'])


movie_transactions = (
    positive_ratings
    .groupby('userId')['title']
    .apply(lambda x: sorted(list(set(x))))
    .reset_index()
)

movie_transactions.columns = ['userId', 'movies']


movie_transactions = movie_transactions[movie_transactions['movies'].apply(len) >= 2]

print("Number of users / transactions:", movie_transactions.shape[0])
print("Example transaction:")
movie_transactions.head()

In [ ]:

movie_frequency = positive_ratings['title'].value_counts()


TOP_N_MOVIES = 300
top_movies = set(movie_frequency.head(TOP_N_MOVIES).index)


filtered_transactions = movie_transactions['movies'].apply(
    lambda movies: [movie for movie in movies if movie in top_movies]
)


filtered_transactions = filtered_transactions[filtered_transactions.apply(len) >= 2].tolist()

print("Number of filtered transactions:", len(filtered_transactions))
print("Number of unique movies used:", len(top_movies))
print("Sample filtered transaction:")
filtered_transactions[0][:10]

In [ ]:

te = TransactionEncoder()
encoded_array = te.fit(filtered_transactions).transform(filtered_transactions)

encoded_movies = pd.DataFrame(encoded_array, columns=te.columns_)

print("Encoded data shape:", encoded_movies.shape)
encoded_movies.head()

In [ ]:
from mlxtend.frequent_patterns import fpgrowth
import pandas as pd


encoded_movies = encoded_movies.astype(bool)



if encoded_movies.shape[0] > 20000:
    encoded_movies_sample = encoded_movies.sample(20000, random_state=42)
else:
    encoded_movies_sample = encoded_movies


MIN_SUPPORT = 0.05


fpgrowth_itemsets = fpgrowth(
    encoded_movies_sample,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=3
)



fpgrowth_itemsets['length'] = fpgrowth_itemsets['itemsets'].apply(len)



fpgrowth_itemsets = fpgrowth_itemsets.sort_values(
    by=['support', 'length'],
    ascending=False
).reset_index(drop=True)

print("Number of frequent itemsets:", fpgrowth_itemsets.shape[0])
display(fpgrowth_itemsets.head(20))

In [ ]:

fpgrowth_combinations = (
    fpgrowth_itemsets[fpgrowth_itemsets['length'] >= 2]
    .sort_values(by=['support', 'length'], ascending=False)
    .reset_index(drop=True)
)

print("Number of frequent movie combinations:", fpgrowth_combinations.shape[0])
fpgrowth_combinations.head(20)

In [ ]:

MIN_CONFIDENCE = 0.25

fpgrowth_rules_raw = association_rules(
    fpgrowth_itemsets,
    metric='confidence',
    min_threshold=MIN_CONFIDENCE
)


fpgrowth_rules = fpgrowth_rules_raw[
    ['antecedents', 'consequents', 'support', 'confidence', 'lift']
].copy()

fpgrowth_rules = fpgrowth_rules[
    (fpgrowth_rules['lift'] > 1)
].reset_index(drop=True)


fpgrowth_rules['antecedents'] = fpgrowth_rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
fpgrowth_rules['consequents'] = fpgrowth_rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))


fpgrowth_rules = fpgrowth_rules.sort_values(
    by=['lift', 'confidence', 'support'],
    ascending=False
).reset_index(drop=True)

print("Number of FP-Growth association rules:", fpgrowth_rules.shape[0])
fpgrowth_rules.head(20)

#  **Model Parameters and Thresholds**

**In this implementation, the following parameters were used to control the quality and performance of the association rule mining process:**


- Minimum Support (min_support = 0.05)
This means that only movie combinations watched by at least 5% of users are considered frequent.

  This helps reduce noise and improves performance by ignoring rare combinations.

- Maximum Itemset Length (max_len = 3)
We limited itemsets to a maximum of 3 movies per combination to avoid excessive
computation and overly complex patterns.

- Minimum Confidence (min_confidence = 0.25)
This means that a rule is accepted only if at least 25% of users who watched Movie A also watched Movie B.

- Lift Threshold (lift > 1)
We kept only rules where lift is greater than 1, ensuring that the relationship between movies is meaningful and not random.

- Top Movies Selection (TOP_N_MOVIES = 300)
Only the top 300 most frequently watched movies were used to reduce dataset size and improve efficiency.

- Sampling (max 20,000 transactions)
If the dataset is large, a sample of 20,000 users is used to speed up processing and avoid memory issues.

In [ ]:
def recommend_movies_fp(movie_title, rules_df, top_n=10):
    '''
    Recommend movies based on FP-Growth association rules.

    Parameters:
    movie_title: movie used as the input
    rules_df: FP-Growth association rules dataframe
    top_n: number of recommendations

    Returns:
    DataFrame with recommended movies and rule metrics
    '''

    recommendations = rules_df[
        rules_df['antecedents'].str.contains(movie_title, case=False, regex=False)
    ].copy()

    recommendations = recommendations.sort_values(
        by=['lift', 'confidence', 'support'],
        ascending=False
    )

    recommendations = recommendations.drop_duplicates(subset=['consequents'])

    return recommendations[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(top_n)



recommend_movies_fp('Toy Story', fpgrowth_rules, top_n=10)

In [ ]:

fpgrowth_itemsets.to_csv('fpgrowth frequent itemsets.csv', index=False)
fpgrowth_combinations.to_csv('fpgrowth frequent movie combinations.csv', index=False)
fpgrowth_rules.to_csv('fpgrowth frequent association rules.csv', index=False)

print(" FP-Growth files saved successfully:")
print("- fpgrowth frequent itemsets.csv")
print("- fpgrowth frequent movie combinations.csv")
print("- fpgrowth frequent association rules.csv")

# **PageRank**


In [ ]:

!pip install networkx --quiet

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:

raw_graph_rules = association_rules(
    fpgrowth_itemsets,
    metric='confidence',
    min_threshold=0.25
)


raw_graph_rules = raw_graph_rules[
    ['antecedents', 'consequents', 'support', 'confidence', 'lift']
].copy()


raw_graph_rules = raw_graph_rules[
    (raw_graph_rules['lift'] > 1) &
    (raw_graph_rules['confidence'] > 0)
].reset_index(drop=True)

print('Number of rules used to build the graph:', raw_graph_rules.shape[0])
raw_graph_rules.head()

In [ ]:

edge_rows = []

for _, row in raw_graph_rules.iterrows():
    antecedents = list(row['antecedents'])
    consequents = list(row['consequents'])

    for source_movie in antecedents:
        for target_movie in consequents:
            if source_movie != target_movie:
                edge_rows.append({
                    'source': source_movie,
                    'target': target_movie,
                    'support': row['support'],
                    'confidence': row['confidence'],
                    'lift': row['lift'],
                    'weight': row['support'] * row['confidence'] * row['lift']
                })

edges_df = pd.DataFrame(edge_rows)


edges_df = (
    edges_df
    .groupby(['source', 'target'], as_index=False)
    .agg({
        'support': 'max',
        'confidence': 'max',
        'lift': 'max',
        'weight': 'sum'
    })
)

print('Number of graph edges:', edges_df.shape[0])
edges_df.sort_values(by='weight', ascending=False).head(20)

In [ ]:

movie_graph = nx.DiGraph()

for _, row in edges_df.iterrows():
    movie_graph.add_edge(
        row['source'],
        row['target'],
        weight=float(row['weight']),
        support=float(row['support']),
        confidence=float(row['confidence']),
        lift=float(row['lift'])
    )

print('Number of nodes / movies:', movie_graph.number_of_nodes())
print('Number of edges / recommendation links:', movie_graph.number_of_edges())
print('Graph density:', nx.density(movie_graph))

In [ ]:

pagerank_scores = nx.pagerank(
    movie_graph,
    alpha=0.85,
    weight='weight'
)

pagerank_df = pd.DataFrame({
    'movie': list(pagerank_scores.keys()),
    'pagerank_score': list(pagerank_scores.values())
})


pagerank_df['in_degree'] = pagerank_df['movie'].apply(lambda x: movie_graph.in_degree(x))
pagerank_df['out_degree'] = pagerank_df['movie'].apply(lambda x: movie_graph.out_degree(x))
pagerank_df['total_degree'] = pagerank_df['in_degree'] + pagerank_df['out_degree']
pagerank_df['weighted_in_degree'] = pagerank_df['movie'].apply(
    lambda x: sum(data['weight'] for _, _, data in movie_graph.in_edges(x, data=True))
)
pagerank_df['weighted_out_degree'] = pagerank_df['movie'].apply(
    lambda x: sum(data['weight'] for _, _, data in movie_graph.out_edges(x, data=True))
)

pagerank_df = pagerank_df.sort_values(
    by='pagerank_score',
    ascending=False
).reset_index(drop=True)

pagerank_df.head(20)

In [ ]:

movie_rating_stats = (
    positive_ratings
    .groupby('title')
    .agg(
        avg_user_rating=('rating', 'mean'),
        liked_by_users=('userId', 'nunique')
    )
    .reset_index()
)

metadata_cols = ['title', 'popularity', 'vote_average', 'vote_count', 'release_year']
metadata_available = [col for col in metadata_cols if col in merged.columns]
movie_metadata = merged[metadata_available].drop_duplicates(subset='title')

pagerank_ranking = (
    pagerank_df
    .merge(movie_rating_stats, left_on='movie', right_on='title', how='left')
    .merge(movie_metadata, left_on='movie', right_on='title', how='left', suffixes=('', '_metadata'))
)


pagerank_ranking = pagerank_ranking.drop(columns=[col for col in ['title', 'title_metadata'] if col in pagerank_ranking.columns])

pagerank_ranking = pagerank_ranking.sort_values(
    by=['pagerank_score', 'weighted_in_degree', 'liked_by_users'],
    ascending=False
).reset_index(drop=True)

pagerank_ranking.head(20)

In [ ]:
top_pagerank_movies = pagerank_ranking.head(20).copy()

top_pagerank_movies[[
    'movie',
    'pagerank_score',
    'in_degree',
    'out_degree',
    'weighted_in_degree',
    'avg_user_rating',
    'liked_by_users'
]].head(20)

In [ ]:
def get_linked_movie_recommendations(movie_title, graph, pagerank_table, top_n=10):


    if movie_title not in graph.nodes:
        return pd.DataFrame({'message': [f'{movie_title} was not found in the movie graph.']})

    outgoing_edges = []

    for _, target, data in graph.out_edges(movie_title, data=True):
        outgoing_edges.append({
            'input_movie': movie_title,
            'recommended_movie': target,
            'edge_weight': data.get('weight', 0),
            'support': data.get('support', 0),
            'confidence': data.get('confidence', 0),
            'lift': data.get('lift', 0)
        })

    recommendations = pd.DataFrame(outgoing_edges)

    if recommendations.empty:
        return pd.DataFrame({'message': [f'No outgoing recommendations found for {movie_title}.']})

    recommendations = recommendations.merge(
        pagerank_table[['movie', 'pagerank_score']],
        left_on='recommended_movie',
        right_on='movie',
        how='left'
    ).drop(columns='movie')

    recommendations = recommendations.sort_values(
        by=['edge_weight', 'pagerank_score', 'confidence', 'lift'],
        ascending=False
    ).reset_index(drop=True)

    return recommendations.head(top_n)


get_linked_movie_recommendations('Toy Story', movie_graph, pagerank_df, top_n=10)

In [ ]:

edges_df.to_csv('movie graph edges.csv', index=False)
pagerank_ranking.to_csv('pagerank movie ranking.csv', index=False)
top_pagerank_movies.to_csv('top pagerank movies.csv', index=False)


nx.write_graphml(movie_graph, 'movie recommendation graph.graphml')

print(' files saved successfully:')
print('- movie graph edges.csv')
print('- pagerank movie ranking.csv ')
print('- top pagerank movies.csv')
print('- movie recommendation graph.graphml')

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx


TOP_GRAPH_NODES = 5
selected_nodes = pagerank_df.head(TOP_GRAPH_NODES)['movie'].tolist()



G = nx.DiGraph()

for u, v, data in movie_graph.edges(data=True):
    if u in selected_nodes and v in selected_nodes:
        G.add_edge(u, v)


def shorten_name(name, max_len=10):
    return name[:max_len] + "..." if len(name) > max_len else name

labels = {node: shorten_name(node) for node in G.nodes()}



plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)

nx.draw(
    G,
    pos,
    with_labels=False,
    node_size=3500,
    node_color='lightblue',
    edge_color='black',
    arrows=True
)


nx.draw_networkx_labels(
    G,
    pos,
    labels=labels,
    font_size=8,
    font_color='black'
)

plt.title("        Simple Movie Graph         ")
plt.axis('on')
plt.show()

# **Visualization**

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import numpy as np

plt.style.use('ggplot')



## TOP MOVIES BY PAGERANK

In [ ]:
top_15_pagerank = pagerank_ranking.head(15)

colors = ['skyblue'] * len(top_15_pagerank)
colors[0] = 'crimson'

plt.figure(figsize=(12,7))

bars = plt.barh(
    top_15_pagerank['movie'][::-1],
    top_15_pagerank['pagerank_score'][::-1],
    color=colors[::-1]
)

plt.xlabel('PageRank Score')
plt.ylabel('Movie')
plt.title('Top 15 Movies by PageRank')


from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor='crimson', label='Top Ranked Movie'),
    Patch(facecolor='skyblue', label='Other Movies')
]

plt.legend(handles=legend_elements)

plt.tight_layout()
plt.show()



## MOST FREQUENT MOVIES

In [ ]:
top_single_movies = (
    fpgrowth_itemsets[fpgrowth_itemsets['length'] == 1]
    .sort_values(by='support', ascending=False)
    .head(15)
)

top_single_movies['movie'] = top_single_movies['itemsets'].apply(lambda x: list(x)[0])

colors = ['skyblue'] * len(top_single_movies)
colors[0] = 'red'

plt.figure(figsize=(12,7))

plt.barh(
    top_single_movies['movie'][::-1],
    top_single_movies['support'][::-1],
    color=colors[::-1]
)

plt.xlabel('Support')
plt.ylabel('Movie')
plt.title('Top 15 Most Frequent Movies')

legend_elements = [
    Patch(facecolor='red', label='Most Frequent Movie'),
    Patch(facecolor='skyblue', label='Other Movies')
]

plt.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

## TOP MOVIE COMBINATIONS

In [ ]:
top_pairs = fpgrowth_combinations.head(15).copy()

top_pairs['pair'] = top_pairs['itemsets'].apply(
    lambda x: ' + '.join(list(x))
)

colors = ['skyblue'] * len(top_pairs)
colors[0] = 'red'

plt.figure(figsize=(14,8))

plt.barh(
    top_pairs['pair'][::-1],
    top_pairs['support'][::-1],
    color=colors[::-1]
)

plt.xlabel('Support')
plt.ylabel('Movie Pair')
plt.title('Top Frequent Movie Combinations')

legend_elements = [
    Patch(facecolor='red', label='Top Combination'),
    Patch(facecolor='skyblue', label='Other Combinations')
]

plt.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

## NETWORK GRAPH (TOP MOVIES)

In [ ]:
top_movies = pagerank_ranking.head(10)['movie'].tolist()

subgraph = movie_graph.subgraph(top_movies)

plt.figure(figsize=(16,12))

pos = nx.spring_layout(
    subgraph,
    seed=42,
    k=1.2
)

node_sizes = []

for node in subgraph.nodes():
    score = pagerank_scores[node]
    node_sizes.append(score * 50000)

nx.draw_networkx_nodes(
    subgraph,
    pos,
    node_size=node_sizes,
    alpha=0.8
)

nx.draw_networkx_edges(
    subgraph,
    pos,
    alpha=0.3,
    arrows=True
)

nx.draw_networkx_labels(
    subgraph,
    pos,
    font_size=9
)

plt.title('Movie Recommendation Network Graph')

plt.axis('off')
plt.tight_layout()
plt.show()

## MOST LIKED MOVIES

In [ ]:


top_liked = (
    pagerank_ranking
    .sort_values(by='liked_by_users', ascending=False)
    .head(15)
)

colors = ['skyblue'] * len(top_liked)
colors[0] = 'red'

plt.figure(figsize=(12,7))

plt.barh(
    top_liked['movie'][::-1],
    top_liked['liked_by_users'][::-1],
    color=colors[::-1]
)

plt.xlabel('Number of Users')
plt.ylabel('Movie')
plt.title('Most Liked Movies')

legend_elements = [
    Patch(facecolor='red', label='Most Liked Movie'),
    Patch(facecolor='skyblue', label='Other Movies')
]

plt.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

# genre distribution

In [ ]:
import plotly.graph_objects as go

genre_series = movies_movielens['genres'].explode()
genre_counts = genre_series.value_counts().head(10)

colors = ['red' if i == 0 else 'skyblue' for i in range(len(genre_counts))]
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=[genre_counts.index[0]],
        y=[genre_counts.values[0]],
        name='Most Common Genre',
        marker_color='red'
    )
)

fig.add_trace(
    go.Bar(
        x=genre_counts.index[1:],
        y=genre_counts.values[1:],
        name='Other Genres',
        marker_color='skyblue'
    )
)

fig.update_layout(
    title='Top Movie Genres',
    xaxis_title='Genre',
    yaxis_title='Count',
    xaxis_tickangle=-45,
    template='plotly_white'
)

fig.show()

# BERT

In [ ]:


print(" Starting BERT section...")

In [ ]:
bert_df = merged[['movieId', 'title', 'overview']].dropna().drop_duplicates()
bert_df = bert_df.reset_index(drop=True)

print("Dataset shape:", bert_df.shape)
bert_df.head()

In [ ]:
import torch
import numpy as np
from transformers import BertTokenizer, BertModel

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("BERT loaded on:", device)

In [ ]:
def compute_embeddings(texts, batch_size=32):

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        last_hidden = outputs.last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).float()

        summed = (last_hidden * mask).sum(dim=1)
        count = mask.sum(dim=1).clamp(min=1e-9)

        batch_embeddings = (summed / count).cpu().numpy()
        all_embeddings.append(batch_embeddings)

    return np.vstack(all_embeddings)

In [ ]:
texts = bert_df['overview'].fillna("").tolist()

movie_embeddings = compute_embeddings(texts)

print("Embeddings shape:", movie_embeddings.shape)

In [ ]:
np.save("movie_bert_embeddings.npy", movie_embeddings)
bert_df.to_csv("bert_movies.csv", index=False)

In [ ]:
print("Sample movie:", bert_df['title'].iloc[0])
print("Embedding preview:", movie_embeddings[0][:10])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

In [ ]:
movie_embeddings = np.load("movie_bert_embeddings.npy")
bert_df = pd.read_csv("bert_movies.csv")

In [ ]:
similarity_matrix = cosine_similarity(movie_embeddings)
print("Similarity matrix shape:", similarity_matrix.shape)

In [ ]:
sim_records = []

for idx, row in bert_df.iterrows():
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [(bert_df.iloc[i]['title'], round(s, 4)) for i, s in scores if i != idx][:5]
    sim_records.append({'movie': row['title'], 'top_similar': scores})

similarity_results = pd.DataFrame(sim_records)
print("Similarity results shape:", similarity_results.shape)
similarity_results.head()

In [ ]:
sim_records = []

for idx, row in bert_df.iterrows():
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [(bert_df.iloc[i]['title'], round(s, 4)) for i, s in scores if i != idx][:5]
    sim_records.append({'movie': row['title'], 'top_similar': scores})

similarity_results = pd.DataFrame(sim_records)
print("Similarity results shape:", similarity_results.shape)
similarity_results.head()

In [ ]:
def recommend_bert(movie_title, top_n=10):
    matches = bert_df[bert_df['title'].str.lower() == movie_title.lower()]
    if matches.empty:
        return f"'{movie_title}' not found."

    idx = matches.index[0]
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    results = pd.DataFrame([{
        'recommended_movie': bert_df.iloc[i]['title'],
        'similarity_score': round(s, 4)
    } for i, s in scores])

    return results

In [ ]:
recommend_bert("Toy Story", top_n=10)

In [ ]:
movie = "Toy Story"

bert_recs = recommend_bert(movie, top_n=10)
fp_recs = recommend_movies_fp(movie, fpgrowth_rules, top_n=10)

print("── BERT Recommendations ──")
print(bert_recs[['recommended_movie', 'similarity_score']].to_string(index=False))

print("\n── FP-Growth Recommendations ──")
print(fp_recs[['consequents', 'confidence', 'lift']].to_string(index=False))

In [ ]:
bert_titles = set(bert_recs['recommended_movie'].tolist())
fp_titles = set(fp_recs['consequents'].tolist())

overlap = bert_titles & fp_titles
print(f"Shared recommendations: {len(overlap)}")
print(overlap)

In [ ]:
all_top_scores = []

for idx in range(len(bert_df)):
    scores = sorted(similarity_matrix[idx], reverse=True)[1:6]
    all_top_scores.append(np.mean(scores))

avg_similarity = np.mean(all_top_scores)
print(f"Average Top-5 Similarity Score: {avg_similarity:.4f}")

In [ ]:
similarity_results.to_csv("bert_similarity_results.csv", index=False)
print("Saved: bert_similarity_results.csv")

In [ ]:
import matplotlib.pyplot as plt

# Define a custom color palette
DARK_BG = '#1e1e1e'
CARD_BG = '#282828'
TEXT_COLOR = '#e0e0e0'
ACCENT1 = '#bb86fc' # Purple
ACCENT2 = '#03dac6' # Teal
ACCENT3 = '#00f260' # Green

sample = recommend_bert("Toy Story", top_n=10)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(CARD_BG)

colors = plt.cm.cool(np.linspace(0.3, 0.9, len(sample)))
bars = ax.barh(sample['recommended_movie'], sample['similarity_score'],
               color=colors, edgecolor='none', height=0.6)

for bar, val in zip(bars, sample['similarity_score']):
    ax.text(bar.get_width() - 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='right', fontsize=9,
            color='white', fontweight='bold')

ax.set_title(' BERT — Top 10 Similar Movies to "Toy Story"',
             fontsize=14, fontweight='bold', color=ACCENT3, pad=15)
ax.set_xlabel('Cosine Similarity Score', color=TEXT_COLOR)
ax.tick_params(colors=TEXT_COLOR)
ax.grid(axis='x', alpha=0.3)
ax.spines[['top','right','left','bottom']].set_visible(False)

plt.tight_layout()
plt.savefig('viz_09_bert_top_similar.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

In [ ]:
# Define a custom color palette
DARK_BG = '#1e1e1e'
CARD_BG = '#282828'
TEXT_COLOR = '#e0e0e0'
ACCENT1 = '#bb86fc' # Purple
ACCENT2 = '#03dac6' # Teal
ACCENT3 = '#00f260' # Green

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(CARD_BG)

ax.hist(all_top_scores, bins=40, color=ACCENT1, edgecolor='none', alpha=0.85)
ax.axvline(avg_similarity, color=ACCENT3, linewidth=2,
           linestyle='--', label=f'Mean: {avg_similarity:.4f}')

ax.set_title('BERT — Distribution of Top-5 Similarity Scores',
             fontsize=14, fontweight='bold', color=ACCENT3, pad=15)
ax.set_xlabel('Average Similarity Score', color=TEXT_COLOR)
ax.set_ylabel('Number of Movies', color=TEXT_COLOR)
ax.legend(facecolor=CARD_BG, labelcolor=TEXT_COLOR)
ax.tick_params(colors=TEXT_COLOR)
ax.grid(alpha=0.3)
ax.spines[['top','right','left','bottom']].set_visible(False)

plt.tight_layout()
plt.savefig('viz_10_bert_similarity_dist.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

In [ ]:
# Define a custom color palette
DARK_BG = '#1e1e1e'
CARD_BG = '#282828'
TEXT_COLOR = '#e0e0e0'
ACCENT1 = '#bb86fc' # Purple
ACCENT2 = '#03dac6' # Teal
ACCENT3 = '#00f260' # Green

bert_only = len(bert_titles - fp_titles)
fp_only = len(fp_titles - bert_titles)
shared = len(overlap)

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(CARD_BG)

categories = ['BERT Only', 'Shared', 'FP-Growth Only']
values = [bert_only, shared, fp_only]
bar_colors = [ACCENT2, ACCENT3, ACCENT1]

bars = ax.bar(categories, values, color=bar_colors, edgecolor='none', width=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(val), ha='center', va='bottom', fontsize=12,
            color=TEXT_COLOR, fontweight='bold')

ax.set_title('BERT vs FP-Growth — Recommendation Overlap',
             fontsize=14, fontweight='bold', color=ACCENT3, pad=15)
ax.set_ylabel('Number of Movies', color=TEXT_COLOR)
ax.tick_params(colors=TEXT_COLOR)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right','left','bottom']].set_visible(False)

plt.tight_layout()
plt.savefig('viz_11_bert_vs_fp_overlap.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

# TOP SIMILAR MOVIES BAR CHART

In [ ]:

movie_name = "Toy Story"

bert_vis = recommend_bert(movie_name, top_n=10)

colors = ['lightblue'] * len(bert_vis)
colors[0] = 'darkblue'

plt.figure(figsize=(12,7))

plt.barh(
    bert_vis['recommended_movie'][::-1],
    bert_vis['similarity_score'][::-1],
    color=colors[::-1]
)

plt.xlabel('Similarity Score')
plt.ylabel('Recommended Movie')

plt.title(f'Top Similar Movies to "{movie_name}" using BERT')

from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor='darkblue', label='Most Similar Movie'),
    Patch(facecolor='lightblue', label='Other Similar Movies')
]

plt.legend(handles=legend_elements)

plt.tight_layout()
plt.show()

# SIMILARITY SCORE DISTRIBUTION

In [ ]:
all_similarity_scores = []

for idx in range(len(similarity_matrix)):

    scores = similarity_matrix[idx]

    scores = sorted(scores, reverse=True)[1:6]

    all_similarity_scores.extend(scores)

plt.figure(figsize=(10,6))

plt.hist(
    all_similarity_scores,
    bins=30
)

plt.xlabel('Similarity Score')
plt.ylabel('Frequency')

plt.title('Distribution of BERT Similarity Scores')

plt.tight_layout()
plt.show()